# Task 1 v2: Learn `u` and `gradU` (Instantaneous Surrogate)

This notebook implements the mode you requested:
- input at time `t`: particle state `x_t`
- output at time `t`: `[u_t, gradU_t]`
- FLOWUnsteady still does O(N) state integration and time marching
- the ML model replaces only expensive velocity/gradient evaluation


In [ ]:
from pathlib import Path
import json
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

NOTEBOOK_VERSION = 'task1_ugradu_v1_2026-05-09'
print('Notebook version:', NOTEBOOK_VERSION)

CWD = Path.cwd().resolve()
if (CWD / 'final-2' / 'output').exists():
    BASE = CWD / 'final-2'
elif (CWD.name == 'notebooks') and (CWD.parent / 'output').exists():
    BASE = CWD.parent
else:
    BASE = CWD

DATA_PATH = BASE / 'output' / 'particle_ugradu_dataset.npz'
OUT_DIR = BASE / 'output' / 'task1_ugradu_training'
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Base:', BASE)
print('DATA_PATH:', DATA_PATH)
print('Exists?:', DATA_PATH.exists())
print('Device:', DEVICE)
if not DATA_PATH.exists():
    raise FileNotFoundError('Run preprocess_data.py with TASK1_TARGET_MODE="ugradu" first')


In [ ]:
# Load dataset

ds = np.load(DATA_PATH, allow_pickle=True)
X = ds['inputs_t_norm'].astype(np.float32)
Y = ds['targets_ugradu_norm'].astype(np.float32)
X_raw = ds['inputs_t'].astype(np.float32)
Y_raw = ds['targets_ugradu'].astype(np.float32)

feature_names = [str(x) for x in ds['feature_names'].tolist()]
target_names = [str(x) for x in ds['target_names'].tolist()]

frame_ranges = list(ds['frame_ranges'])
frame_contexts = list(ds['frame_contexts'])
train_frame_ids = ds['train_frame_ids'].astype(np.int64)
val_frame_ids = ds['val_frame_ids'].astype(np.int64)
test_frame_ids = ds['test_frame_ids'].astype(np.int64)

train_rows = ds['train_rows'].astype(np.int64)
val_rows = ds['val_rows'].astype(np.int64)
test_rows = ds['test_rows'].astype(np.int64)

train_cases = [str(x) for x in ds['train_cases'].tolist()]
val_cases = [str(x) for x in ds['val_cases'].tolist()]
test_cases = [str(x) for x in ds['test_cases'].tolist()]

out_mean = ds['out_mean'].astype(np.float32)
out_std = ds['out_std'].astype(np.float32)

print('inputs_t_norm shape      :', X.shape)
print('targets_ugradu_norm shape:', Y.shape)
print('n_frames                 :', len(frame_ranges))
print('feature_names            :', feature_names)
print('target_names             :', target_names)
print('train/val/test frames    :', len(train_frame_ids), len(val_frame_ids), len(test_frame_ids))
print('train/val/test rows      :', len(train_rows), len(val_rows), len(test_rows))
print('train/val/test cases     :', train_cases, val_cases, test_cases)

geom_candidates = [k for k in feature_names if k.startswith('geom_')]
print('geometry channels in dataset:', geom_candidates)


In [ ]:
# Dataset class: one sample = one frame (variable particle count)

class FrameDataset(Dataset):
    def __init__(self, X, Y, frame_ranges, frame_ids):
        self.X = X
        self.Y = Y
        self.frame_ranges = frame_ranges
        self.frame_ids = [int(i) for i in frame_ids]

    def __len__(self):
        return len(self.frame_ids)

    def __getitem__(self, idx):
        fid = self.frame_ids[idx]
        case, fr, s, e, n = self.frame_ranges[fid]
        s, e = int(s), int(e)
        x = torch.from_numpy(self.X[s:e])
        y = torch.from_numpy(self.Y[s:e])
        meta = {'frame_id': fid, 'case': str(case), 'fr': str(fr), 'n': int(n)}
        return x, y, meta


def collate_frame(batch):
    xs, ys, ms = zip(*batch)
    return list(xs), list(ys), list(ms)

train_ds = FrameDataset(X, Y, frame_ranges, train_frame_ids)
val_ds = FrameDataset(X, Y, frame_ranges, val_frame_ids)
test_ds = FrameDataset(X, Y, frame_ranges, test_frame_ids)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=collate_frame)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=collate_frame)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=collate_frame)

print('dataset lens:', len(train_ds), len(val_ds), len(test_ds))


In [ ]:
# Model: GNO for instantaneous u/gradU regression
import inspect

try:
    from neuralop.layers.gno_block import GNOBlock
except Exception as exc:
    raise RuntimeError('Need neuralop with GNOBlock') from exc


def rel_l2(pred, tgt, eps=1e-12):
    d = (pred - tgt).reshape(pred.shape[0], -1)
    t = tgt.reshape(tgt.shape[0], -1)
    return (torch.linalg.norm(d, dim=1) / torch.linalg.norm(t, dim=1).clamp_min(eps)).mean()


class UGradUGNO(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=96, n_layers=3, radius=0.12):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(in_dim, hidden), nn.GELU(), nn.Linear(hidden, hidden))

        sig = inspect.signature(GNOBlock.__init__)
        p = sig.parameters
        extra = {}
        if 'use_torch_scatter_reduce' in p:
            extra['use_torch_scatter_reduce'] = False
        if 'use_open3d_neighbor_search' in p:
            extra['use_open3d_neighbor_search'] = False

        self.blocks = nn.ModuleList([
            GNOBlock(
                in_channels=hidden,
                out_channels=hidden,
                coord_dim=3,
                radius=radius,
                transform_type='linear',
                reduction='mean',
                pos_embedding_type='transformer',
                pos_embedding_channels=12,
                channel_mlp_layers=[hidden, hidden, hidden],
                **extra,
            ) for _ in range(n_layers)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, out_dim))

    def forward(self, x):
        pos = x[:, :3]  # x,y,z
        h = self.enc(x)
        for blk, norm in zip(self.blocks, self.norms):
            u = blk(y=pos, x=pos, f_y=h)
            if u.ndim == 3 and u.shape[0] == 1:
                u = u.squeeze(0)
            h = norm(h + u)
        return self.head(h)


model = UGradUGNO(X.shape[1], Y.shape[1], hidden=96, n_layers=3, radius=0.12).to(DEVICE)
print('params:', sum(p.numel() for p in model.parameters() if p.requires_grad))


In [ ]:
# Optimizer + training settings
opt_cls = None
try:
    import neuralop.training as nt
    opt_cls = getattr(nt, 'AdamW', None) or getattr(nt, 'Adam', None)
except Exception:
    pass
if opt_cls is None:
    opt_cls = torch.optim.AdamW

opt = opt_cls(model.parameters(), lr=1e-3, weight_decay=1e-6)
sch = torch.optim.lr_scheduler.StepLR(opt, step_size=100, gamma=0.5)

EPOCHS = 80
MAX_NODES = 1024
FRAMES_PER_EPOCH = 96
VAL_FRAMES_LIMIT = 48
PRINT_EVERY_STEPS = 8

print('config:', {
    'EPOCHS': EPOCHS,
    'MAX_NODES': MAX_NODES,
    'FRAMES_PER_EPOCH': FRAMES_PER_EPOCH,
    'VAL_FRAMES_LIMIT': VAL_FRAMES_LIMIT,
})


In [ ]:
# Training + validation
history = []
best_val = np.inf
best_state = None


def eval_loader(loader, max_frames=48):
    model.eval()
    rels, mses = [], []
    vel_rels, grad_rels = [], []

    with torch.no_grad():
        for b, (xs, ys, ms) in enumerate(loader):
            if b >= max_frames:
                break
            x, y = xs[0], ys[0]
            if x.shape[0] > MAX_NODES:
                idx = torch.randperm(x.shape[0])[:MAX_NODES]
                x, y = x[idx], y[idx]

            x = x.to(DEVICE)
            y = y.to(DEVICE)
            p = model(x)

            rels.append(float(rel_l2(p.unsqueeze(0), y.unsqueeze(0)).item()))
            mses.append(float(torch.mean((p - y) ** 2).item()))

            vel_rels.append(float(rel_l2(p[:, :3].unsqueeze(0), y[:, :3].unsqueeze(0)).item()))
            grad_rels.append(float(rel_l2(p[:, 3:].unsqueeze(0), y[:, 3:].unsqueeze(0)).item()))

    return {
        'rel_l2': float(np.mean(rels)),
        'mse': float(np.mean(mses)),
        'vel_rel_l2': float(np.mean(vel_rels)),
        'grad_rel_l2': float(np.mean(grad_rels)),
    }

print('Smoke test...')
x0, y0, m0 = train_ds[0]
if x0.shape[0] > MAX_NODES:
    idx = torch.randperm(x0.shape[0])[:MAX_NODES]
    x0, y0 = x0[idx], y0[idx]
x0, y0 = x0.to(DEVICE), y0.to(DEVICE)
t0 = time.time()
p0 = model(x0)
loss0 = rel_l2(p0.unsqueeze(0), y0.unsqueeze(0))
loss0.backward()
model.zero_grad(set_to_none=True)
print(f'Smoke OK | n={x0.shape[0]} | sec={time.time()-t0:.2f}')

for ep in range(1, EPOCHS + 1):
    t_ep = time.time()
    model.train()
    losses = []

    n_train = len(train_ds)
    use_n = min(FRAMES_PER_EPOCH, n_train)
    picks = np.random.choice(n_train, size=use_n, replace=False)

    for step, i in enumerate(picks, start=1):
        x, y, meta = train_ds[int(i)]
        if x.shape[0] > MAX_NODES:
            idx = torch.randperm(x.shape[0])[:MAX_NODES]
            x, y = x[idx], y[idx]

        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        p = model(x)
        loss = rel_l2(p.unsqueeze(0), y.unsqueeze(0))
        loss.backward()
        opt.step()

        losses.append(float(loss.item()))
        if step % PRINT_EVERY_STEPS == 0:
            print(f'[ep {ep:03d}] step {step:03d}/{use_n} train_rel={loss.item():.6f}', flush=True)

    sch.step()

    train_rel = float(np.mean(losses))
    val = eval_loader(val_loader, max_frames=VAL_FRAMES_LIMIT)
    test = eval_loader(test_loader, max_frames=VAL_FRAMES_LIMIT)

    row = {
        'epoch': ep,
        'train_rel_l2': train_rel,
        'val_rel_l2': val['rel_l2'],
        'val_mse': val['mse'],
        'val_vel_rel_l2': val['vel_rel_l2'],
        'val_grad_rel_l2': val['grad_rel_l2'],
        'test_rel_l2': test['rel_l2'],
        'test_mse': test['mse'],
        'test_vel_rel_l2': test['vel_rel_l2'],
        'test_grad_rel_l2': test['grad_rel_l2'],
        'epoch_sec': float(time.time() - t_ep),
    }
    history.append(row)

    if val['rel_l2'] < best_val:
        best_val = val['rel_l2']
        best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}

    print(
        f"[epoch {ep:03d}] train={train_rel:.6f} val={val['rel_l2']:.6f} "
        f"val_vel={val['vel_rel_l2']:.6f} val_grad={val['grad_rel_l2']:.6f} "
        f"sec={row['epoch_sec']:.1f}",
        flush=True,
    )

    (OUT_DIR / 'history.json').write_text(json.dumps(history, indent=2))

if best_state is not None:
    model.load_state_dict(best_state)

torch.save(model.state_dict(), OUT_DIR / 'best_task1_ugradu_gno.pt')
summary = {
    'best_val_rel_l2': float(best_val),
    'epochs': EPOCHS,
    'device': str(DEVICE),
    'train_frames': len(train_ds),
    'val_frames': len(val_ds),
    'test_frames': len(test_ds),
}
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2))
print('Saved summary:', json.dumps(summary, indent=2))


In [ ]:
# Evaluation snapshots + plots

def denorm_out(y_norm):
    return y_norm * out_std + out_mean

# Pick one test frame for plotting
x_norm, y_norm, meta = test_ds[0]
x_raw = X_raw[int(frame_ranges[test_frame_ids[0]][2]):int(frame_ranges[test_frame_ids[0]][3])]

if x_norm.shape[0] > MAX_NODES:
    idx = torch.randperm(x_norm.shape[0])[:MAX_NODES]
    x_norm = x_norm[idx]
    y_norm = y_norm[idx]
    x_raw = x_raw[idx.numpy()]

with torch.no_grad():
    y_pred_norm = model(x_norm.to(DEVICE)).cpu().numpy()

y_true = denorm_out(y_norm.numpy())
y_pred = denorm_out(y_pred_norm)

# 1) loss curves
plt.figure(figsize=(8,4))
plt.plot([h['epoch'] for h in history], [h['train_rel_l2'] for h in history], label='train_rel_l2')
plt.plot([h['epoch'] for h in history], [h['val_rel_l2'] for h in history], label='val_rel_l2')
plt.plot([h['epoch'] for h in history], [h['val_vel_rel_l2'] for h in history], label='val_vel_rel_l2')
plt.plot([h['epoch'] for h in history], [h['val_grad_rel_l2'] for h in history], label='val_grad_rel_l2')
plt.grid(alpha=0.3); plt.legend(); plt.xlabel('epoch'); plt.ylabel('error')
plt.title('Task1-u/gradU training curves')
plt.tight_layout(); plt.show()

# 2) histogram of output errors
err = y_pred - y_true
plt.figure(figsize=(7,4))
plt.hist(np.linalg.norm(err[:, :3], axis=1), bins=60, alpha=0.7, label='velocity error norm')
plt.hist(np.linalg.norm(err[:, 3:], axis=1), bins=60, alpha=0.7, label='gradU error norm')
plt.legend(); plt.xlabel('error norm'); plt.ylabel('count'); plt.title('Prediction error histograms')
plt.tight_layout(); plt.show()

# 3) spatial plots on x-z plane: velocity magnitude, gamma magnitude, sigma
x = x_raw[:, 0]
z = x_raw[:, 2]
gamma_mag = np.linalg.norm(x_raw[:, 3:6], axis=1)
sigma = x_raw[:, 6]
vel_true = np.linalg.norm(y_true[:, :3], axis=1)
vel_pred = np.linalg.norm(y_pred[:, :3], axis=1)

fig, axs = plt.subplots(2, 3, figsize=(15, 9))
sc = axs[0,0].scatter(x, z, c=vel_true, s=2, cmap='turbo'); axs[0,0].set_title('True |u|'); fig.colorbar(sc, ax=axs[0,0])
sc = axs[1,0].scatter(x, z, c=vel_pred, s=2, cmap='turbo'); axs[1,0].set_title('Pred |u|'); fig.colorbar(sc, ax=axs[1,0])

sc = axs[0,1].scatter(x, z, c=gamma_mag, s=2, cmap='viridis'); axs[0,1].set_title('Input |Gamma|'); fig.colorbar(sc, ax=axs[0,1])
sc = axs[1,1].scatter(x, z, c=np.abs(vel_pred-vel_true), s=2, cmap='magma'); axs[1,1].set_title('|u| abs error'); fig.colorbar(sc, ax=axs[1,1])

sc = axs[0,2].scatter(x, z, c=sigma, s=2, cmap='plasma'); axs[0,2].set_title('Input sigma'); fig.colorbar(sc, ax=axs[0,2])
sc = axs[1,2].scatter(x, z, c=np.linalg.norm(err[:,3:], axis=1), s=2, cmap='magma'); axs[1,2].set_title('gradU error norm'); fig.colorbar(sc, ax=axs[1,2])

for a in axs.ravel():
    a.set_xlabel('x'); a.set_ylabel('z')
fig.tight_layout(); plt.show()

# Save arrays
np.savez_compressed(
    OUT_DIR / 'test_frame_predictions.npz',
    x_raw=x_raw,
    y_true=y_true,
    y_pred=y_pred,
    error=err,
    feature_names=np.asarray(feature_names, dtype=object),
    target_names=np.asarray(target_names, dtype=object),
)
